# 01 CERF raster extraction
Original raster processing retained. Inputs: `raw data/CERF/`. Outputs: `processed/`.


Run this notebook in VS Code Jupyter with the repository open. The setup locates the repository from the Jupyter working directory or `REPO_DIR_OVERRIDE`.


In [ ]:
from pathlib import Path
import subprocess

# Notebooks live in scripts/; the repository contains raw data/, run_folder/, and processed/.
REPO_DIR_OVERRIDE = None  # Optional: full local repository path if Jupyter runs elsewhere

def find_repo():
    if REPO_DIR_OVERRIDE is not None:
        root = Path(REPO_DIR_OVERRIDE).expanduser().resolve()
    else:
        cwd = Path.cwd().resolve()
        candidates = [cwd, *cwd.parents]
        root = next((p for p in candidates if (p / 'run_folder').is_dir() and (p / 'raw data').is_dir()), None)
        if root is None:
            result = subprocess.run(['git', 'rev-parse', '--show-toplevel'], cwd=cwd, capture_output=True, text=True)
            if result.returncode == 0:
                root = Path(result.stdout.strip()).resolve()
        if root is None:
            raise FileNotFoundError('Jupyter is outside the repository. Open the notebook in VS Code from the repo, or set REPO_DIR_OVERRIDE.')
    if not (root / 'run_folder').is_dir() or not (root / 'raw data').is_dir():
        raise FileNotFoundError(f'Expected raw data/ and run_folder/ under {root}')
    return root

project_dir = find_repo()
RAW_DIR = project_dir / 'raw data'
RUN_DIR = project_dir / 'run_folder'
OUTPUT_DIR = project_dir / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_input(filename=None, folder=None, search_dir=None):
    base = search_dir or RUN_DIR
    if (filename is None) == (folder is None):
        raise ValueError('Specify exactly one of filename or folder')
    matches = sorted(p for p in base.rglob('*') if (p.is_file() and p.name.lower() == filename.lower()) if filename is not None) if filename else sorted(p for p in base.rglob('*') if p.is_dir() and p.name.lower() == folder.lower())
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one {filename or folder} under {base}; found {len(matches)}: {matches[:10]}')
    return matches[0]

print('Repository:', project_dir)
print('Raw rasters:', RAW_DIR / 'CERF')
print('Cleaning inputs:', RUN_DIR)
print('Processed outputs:', OUTPUT_DIR)


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
from rasterio.transform import xy

DATA_DIR = RAW_DIR / 'CERF'
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'CERF raster directory missing: {DATA_DIR}')
tif_files = sorted(DATA_DIR.rglob('*.tif'))
tif_files_raw = tif_files
if not tif_files:
    raise FileNotFoundError(f'No .tif files under {DATA_DIR}; check Git LFS download.')
print('CERF rasters:', len(tif_files))
for p in tif_files: print(p.relative_to(DATA_DIR))

In [ ]:
# Profile rasters safely by block
def profile_raster(raster_path):
    valid_count = 0
    nodata_count = 0
    min_value = None
    max_value = None
    value_sum = 0.0

    with rasterio.open(raster_path) as src:
        for _, window in src.block_windows(1):
            data = src.read(1, window=window, masked=True)
            valid = data.compressed()

            valid_count += valid.size
            nodata_count += data.size - valid.size

            if valid.size == 0:
                continue

            block_min = float(valid.min())
            block_max = float(valid.max())
            min_value = block_min if min_value is None else min(min_value, block_min)
            max_value = block_max if max_value is None else max(max_value, block_max)
            value_sum += valid.astype("float64").sum()

        mean_value = value_sum / valid_count if valid_count else np.nan

        return {
            "file_name": raster_path.name,
            "file_size_mb": round(raster_path.stat().st_size / (1024**2), 2),
            "width": src.width,
            "height": src.height,
            "bands": src.count,
            "dtype": src.dtypes[0],
            "nodata": src.nodata,
            "crs": str(src.crs),
            "resolution_x": src.res[0],
            "resolution_y": src.res[1],
            "valid_cell_count": valid_count,
            "nodata_cell_count": nodata_count,
            "min_value": min_value,
            "max_value": max_value,
            "mean_value": mean_value,
        }

raster_profile = pd.DataFrame([profile_raster(file) for file in tif_files])
raster_profile


In [ ]:
# Load U.S. county boundaries
COUNTY_URL = "https://www2.census.gov/geo/tiger/TIGER2024/COUNTY/tl_2024_us_county.zip"
counties = gpd.read_file(COUNTY_URL)[
    ["GEOID", "STATEFP", "COUNTYFP", "NAME", "NAMELSAD", "geometry"]
].copy()

counties = counties.rename(columns={"GEOID": "county_fips", "NAME": "county_name"})
counties["STATEFP"] = counties["STATEFP"].astype(str).str.zfill(2)
counties["COUNTYFP"] = counties["COUNTYFP"].astype(str).str.zfill(3)
counties["county_fips"] = counties["county_fips"].astype(str).str.zfill(5)

print(f"Loaded {len(counties):,} counties")
print("County CRS:", counties.crs)
counties.head()

In [ ]:
# County aggregation rules
raster_config = {
    "commercial_electricity_rates_dollar_per_kwh.tif": {"type": "continuous", "stats": ["mean", "median"]},
    "industrial_electricity_rates_dollar_per_kwh.tif": {"type": "continuous", "stats": ["mean", "median"]},
    "distance_to_substation.tif": {"type": "continuous", "stats": ["mean", "min", "median"]},
    "land_value_dollar_per_sqft.tif": {"type": "continuous", "stats": ["mean", "median"]},
    "sales_tax_rate.tif": {"type": "continuous", "stats": ["mean", "median"]},
    "real_property_tax_rate.tif": {"type": "continuous", "stats": ["mean", "median"]},
    "personal_property_tax_rate.tif": {"type": "continuous", "stats": ["mean", "median"]},
    "water_cooling_fraction.tif": {"type": "continuous", "stats": ["mean", "median"]},
    "mechanical_cooling_fraction.tif": {"type": "continuous", "stats": ["mean", "median"]},
    "composite_siting_suitability.tif": {"type": "continuous", "stats": ["mean", "median", "max"]},
    "state_siting_region.tif": {"type": "categorical", "stats": ["majority"]},
    "data_center_market_locations.tif": {"type": "continuous", "stats": ["max","sum"]},
}

print(f"Configured {len(raster_config)} rasters")


In [ ]:
# Add county centroids
counties_geo = counties.to_crs("EPSG:4326").copy()
counties_projected = counties.to_crs("EPSG:5070")
centroids = counties_projected.geometry.centroid
centroids_geo = gpd.GeoSeries(centroids, crs="EPSG:5070").to_crs("EPSG:4326")

counties_geo["centroid_longitude"] = centroids_geo.x
counties_geo["centroid_latitude"] = centroids_geo.y

counties_geo[[
    "county_fips",
    "county_name",
    "centroid_longitude",
    "centroid_latitude",
]].head()


## County aggregation
The original county aggregation loop is enabled.

In [ ]:
# Start with the county table
cerf_county = counties_geo.copy()

# Process each raster
for raster_path in tif_files:

    file_name = raster_path.name

    # Skip rasters not defined in config
    if file_name not in raster_config:
        print(f"Skipping: {file_name}")
        continue

    print(f"Processing: {file_name}")

    config = raster_config[file_name]

    with rasterio.open(raster_path) as src:

        # Match counties to raster CRS
        counties_raster = counties_geo.to_crs(src.crs)

        # Continuous rasters
        if config["type"] == "continuous":

            stats = zonal_stats(
                counties_raster.geometry,
                raster_path,
                stats=config["stats"],
                nodata=src.nodata,
                all_touched=False
            )

            base_name = raster_path.stem

            for stat in config["stats"]:

                column_name = f"{base_name}_{stat}"

                cerf_county[column_name] = [
                    row.get(stat, np.nan)
                    for row in stats
                ]

        # Categorical rasters
        elif config["type"] == "categorical":

            stats = zonal_stats(
                counties_raster.geometry,
                raster_path,
                stats=["majority"],
                nodata=src.nodata,
                all_touched=False
            )

            base_name = raster_path.stem

            column_name = f"{base_name}_majority"

            cerf_county[column_name] = [
                row.get("majority", np.nan)
                for row in stats
            ]

print("Finished extracting raster values.")

In [ ]:
print("Rows:", cerf_county.shape[0])
print("Columns:", cerf_county.shape[1])

cerf_county.head()

In [ ]:
cerf_county.columns.tolist()

In [ ]:
cerf_tabular = cerf_county.drop(
    columns=["geometry"]
).copy()

cerf_tabular.head()

In [ ]:
county_output = OUTPUT_DIR / 'stage1_cerf_extraction' / 'cerf_county_tabular.csv'
county_output.parent.mkdir(parents=True, exist_ok=True)
cerf_tabular.to_csv(county_output, index=False)
print('Saved county table:', county_output, '| rows:', len(cerf_tabular))

# CERF County-Level Dataset Column Key

## Geographic and Identifier Columns

- **county_fips**  
  Five-digit county FIPS code. This is the unique geographic identifier for each county and is useful for joining to other county-level datasets.

- **STATEFP**  
  Two-digit state FIPS code.

- **COUNTYFP**  
  Three-digit county FIPS code within the state. Combined with `STATEFP`, it forms `county_fips`.

- **county_name**  
  Short county name, such as `Cuming` or `Lancaster`.

- **NAMELSAD**  
  Full county name, such as `Cuming County`.

- **geometry**  
  Full polygon boundary of the county. Used for GIS operations and county-level mapping.

- **centroid_longitude**  
  Longitude of the approximate geographic center of the county. Useful for map points and labels.

- **centroid_latitude**  
  Latitude of the approximate geographic center of the county. Useful for map points and labels.

---

## Commercial Electricity Rate

- **commercial_electricity_rates_dollar_per_kwh_mean**  
  Average commercial electricity price, in dollars per kWh, across raster cells within the county.

- **commercial_electricity_rates_dollar_per_kwh_median**  
  Middle commercial electricity rate within the county. Less affected by unusually high or low values than the mean.

---

## Industrial Electricity Rate

- **industrial_electricity_rates_dollar_per_kwh_mean**  
  Average industrial electricity price, in dollars per kWh, across the county.

- **industrial_electricity_rates_dollar_per_kwh_median**  
  Middle industrial electricity rate within the county.

---

## Distance to Substation

- **distance_to_substation_mean**  
  Average distance to a substation across the county.

- **distance_to_substation_min**  
  Smallest distance to a substation found within the county. Represents the closest raster location to substation infrastructure.

- **distance_to_substation_median**  
  Middle distance-to-substation value across the county.

---

## Land Value

- **land_value_dollar_per_sqft_mean**  
  Average land value per square foot across the county.

- **land_value_dollar_per_sqft_median**  
  Middle land value per square foot within the county.

---

## Sales Tax Rate

- **sales_tax_rate_mean**  
  Average sales-tax rate represented across the county.

- **sales_tax_rate_median**  
  Middle sales-tax rate represented within the county.

---

## Real Property Tax Rate

- **real_property_tax_rate_mean**  
  Average real-property tax rate across the county.

- **real_property_tax_rate_median**  
  Middle real-property tax rate within the county.

---

## Personal Property Tax Rate

- **personal_property_tax_rate_mean**  
  Average personal-property tax rate represented across the county.

- **personal_property_tax_rate_median**  
  Middle personal-property tax rate within the county.

---

## Water Cooling Fraction

- **water_cooling_fraction_mean**  
  Average water-cooling fraction across the county.

- **water_cooling_fraction_median**  
  Middle water-cooling fraction across the county.

---

## Mechanical Cooling Fraction

- **mechanical_cooling_fraction_mean**  
  Average mechanical-cooling fraction across the county.

- **mechanical_cooling_fraction_median**  
  Middle mechanical-cooling fraction across the county.

---

## Composite Siting Suitability

- **composite_siting_suitability_mean**  
  Average CERF composite siting-suitability value across the county.

- **composite_siting_suitability_median**  
  Middle siting-suitability value within the county.

- **composite_siting_suitability_max**  
  Highest siting-suitability value found anywhere within the county. Useful for identifying counties that contain at least one highly suitable area.

---

## State Siting Region

- **state_siting_region_majority**  
  Most common state siting-region category found within the county.

  This raster is categorical, so the majority category is used instead of calculating an average.

---

## Data Center Market Locations

- **data_center_market_locations_majority**  
  Most common data-center-market-location category found within the county.

  This raster is categorical, so the majority category is used instead of calculating an average.

---

## Statistic Key

- **mean**  
  Average of all valid raster-cell values inside the county.

- **median**  
  Middle value of the raster cells inside the county. Less sensitive to extreme values than the mean.

- **min**  
  Smallest raster value found inside the county.

- **max**  
  Largest raster value found inside the county.

- **majority**  
  Most frequently occurring category inside the county.

---

## Modeling Note

The geographic and identifier fields such as `county_fips`, county names, latitude, longitude, and `geometry` are mainly used for identification, joining datasets, and mapping.

The electricity, distance, land value, tax, cooling, market

## Raw raster cell extraction
This can create hundreds of millions of rows and many Parquet files. Confirm disk space first.

In [ ]:
# -----------------------------
# Check raster alignment
# -----------------------------

reference_path = tif_files_raw[0]

with rasterio.open(reference_path) as ref:
    reference_crs = ref.crs
    reference_transform = ref.transform
    reference_width = ref.width
    reference_height = ref.height

same_grid = True

for raster_path in tif_files_raw:

    with rasterio.open(raster_path) as src:

        matches = (
            src.crs == reference_crs
            and src.transform == reference_transform
            and src.width == reference_width
            and src.height == reference_height
        )

        print(
            raster_path.name,
            "MATCH" if matches else "DIFFERENT GRID"
        )

        if not matches:
            same_grid = False

print()
print("All rasters aligned:", same_grid)

In [ ]:
if not same_grid:
    raise ValueError('Raster grids are not aligned; original raw extraction requires alignment.')
RAW_OUTPUT_DIR = OUTPUT_DIR / 'cerf_non_aggregated_tabular'
RAW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Raw output:', RAW_OUTPUT_DIR)

In [ ]:
sources = {
    path.stem: rasterio.open(path)
    for path in tif_files_raw
}

reference = sources[reference_path.stem]

part_number = 0
saved_files = []

for _, window in reference.block_windows(1):

    rows, cols = np.indices(
        (
            int(window.height),
            int(window.width)
        )
    )

    rows = rows + int(window.row_off)
    cols = cols + int(window.col_off)

    xs, ys = xy(
        reference.transform,
        rows,
        cols,
        offset="center"
    )

    block = pd.DataFrame({
        "raster_row": rows.ravel(),
        "raster_col": cols.ravel(),
        "longitude": np.asarray(xs).ravel(),
        "latitude": np.asarray(ys).ravel()
    })

    valid_any = np.zeros(
        len(block),
        dtype=bool
    )

    for name, src in sources.items():

        data = src.read(
            1,
            window=window,
            masked=True
        )

        values = (
            data
            .astype("float64")
            .filled(np.nan)
            .ravel()
        )

        block[name] = values

        valid_any = valid_any | ~np.isnan(values)

    # Remove locations with no raster data
    block = block.loc[valid_any].copy()

    if block.empty:
        continue

    part_path = (
        RAW_OUTPUT_DIR /
        f"cerf_raw_cells_part_{part_number:05d}.parquet"
    )

    block.to_parquet(
        part_path,
        index=False
    )

    saved_files.append(part_path)

    part_number += 1

    if part_number % 100 == 0:
        print(f"Saved {part_number} blocks")


for src in sources.values():
    src.close()

print()
print(f"Finished. Created {len(saved_files)} parquet files.")

In [ ]:
print('Raw Parquet parts:', len(list(RAW_OUTPUT_DIR.glob('cerf_raw_cells_part_*.parquet'))))
print('Raw output directory:', RAW_OUTPUT_DIR)